# cap10_joint_contract_checkpoint_drift_selector_v1

Purpose: convert the previous BC-good / PPO-drift result into a checkpoint selector. The selector treats the BC initialization as the safety baseline and only lets PPO checkpoints win when they satisfy the return/drawdown gate and do not worsen drawdown versus BC.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baseline: `cap10_joint_contract_bc_ppo_drift_screen_v1`.
- Role: `diagnose`.
- Changed layer: `model_selection` / `checkpoint_selection` only.
- This notebook reruns a short diagnostic PPO screen to recover retained checkpoint states in memory, evaluates BC + retained PPO checkpoints, writes metrics, and retains no actor bundle.
- Target gate remains excess >= +30pp and MaxDD <= 15% on full fixed split plus walk-forward windows.

External basis:
- Qlib research workflow separation: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO as the policy-improvement candidate: https://arxiv.org/abs/1707.06347
- Conservative / safe improvement caution: https://arxiv.org/abs/2006.04779 and https://arxiv.org/abs/1712.06924
- Backtest-overfit discipline: https://doi.org/10.21314/JCF.2016.322

Closest historical success: BC initialization already passed slice fit and one later fold (`wf_2023_2024`) reached the +30pp / <=15% target.

Closest historical failure: `cap10_joint_contract_bc_ppo_drift_screen_v1` showed `wf_2022` PPO improved relative return but pushed MaxDD from 18.46% to 37.28%, so naive policy improvement breaks the risk objective.

Why this is not a duplicate: the prior notebook compared only BC vs selected PPO. This notebook evaluates retained checkpoint states and formalizes a drift-penalized selector that can decide whether the next step should be BC reference, drift penalty, or fuller walk-forward validation.


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'cap10_joint_contract_checkpoint_drift_selector_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_joint_contract_bc_ppo_drift_screen_v1'
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_foldlocal_joint_target_contract_v1'
TARGETS_PATH = TARGET_CARD / 'joint_target_contract.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'selection.json'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_SLICE_ACCURACY_MIN = 0.60
REQUIRED_SLICES = {'release_gate_state': 20, 'is_downtrend_state': 100, 'drawdown_control_state': 20}
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1


In [2]:
source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH)
targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    valid_start = pd.Timestamp(spec['valid'][0])
    valid_end = pd.Timestamp(spec['valid'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(train_start, train_end)]
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'rebound_rows': int(fold_targets.get('is_rebound_cluster', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {'rows': coverage.to_dict('records'), 'source_decision': source_selection.get('decision')})
display(coverage)


,fold,target_fold,train_target_rows,valid_overlap_rows,test_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows,rebound_rows,train_start,train_end,valid_start,valid_end,test_start,test_end
0,wf_2022,wf_2022_train,253,252,0,40,118,49,66,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
1,wf_2023_2024,wf_2023_2024_train,756,0,0,40,478,49,66,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


In [3]:
def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    return out

def candidate_loss(row: dict[str, Any]) -> float:
    rel = float(row.get('test_relative_return', float('-inf')))
    dd = float(row.get('test_max_dd', float('inf')))
    rel_delta = float(row.get('relative_delta_vs_bc', float('-inf')))
    dd_delta = float(row.get('max_dd_delta_vs_bc', float('inf')))
    loss = 0.0
    loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / TARGET_RELATIVE_RETURN
    loss += max(0.0, dd - TARGET_MAX_DD) / TARGET_MAX_DD
    loss += max(0.0, DRIFT_RELATIVE_FLOOR_PP - rel_delta) / max(abs(DRIFT_RELATIVE_FLOOR_PP), 1.0)
    loss += max(0.0, dd_delta - DRIFT_MAX_DD_CEILING_PP) / max(DRIFT_MAX_DD_CEILING_PP, 1.0)
    if row.get('candidate_kind') != 'bc_initialization':
        loss += 0.001 * max(float(row.get('iter', 0) or 0), 0.0)
    return float(loss)

def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        if slice_name not in indexed.index:
            row = {'samples': 0, 'accuracy': float('nan'), 'gate': False}
        else:
            item = indexed.loc[slice_name]
            row = {'samples': int(item.get('samples', 0)), 'accuracy': float(item.get('accuracy', float('nan')))}
            row['gate'] = bool(row['samples'] >= min_samples and math.isfinite(row['accuracy']) and row['accuracy'] >= BC_SLICE_ACCURACY_MIN)
        row['min_samples'] = int(min_samples)
        row['min_accuracy'] = BC_SLICE_ACCURACY_MIN
        details[slice_name] = row
        gate = gate and bool(row['gate'])
    return bool(gate), details


In [4]:
init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
obs_dim = observation_dim_for_profile(profile)

candidate_rows = []
selection_rows = []
history_rows = []
bc_slice_frames = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"fold={fold_name} target_fold={spec['target_fold']}")
    set_seed(SEED)
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(train_start, train_end)].copy()
    test_overlap = targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)]
    if fold_targets.empty or not test_overlap.empty:
        selection_rows.append({
            'fold': fold_name,
            'selected_candidate': None,
            'status': 'blocked_or_missing_targets',
            'target_rows': int(len(fold_targets)),
            'test_overlap_rows': int(len(test_overlap)),
        })
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=None,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
    )
    bc_slice_gate, bc_slice_details = slice_gate_from_metrics(bc_slice_metrics)
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    test_price_series = price_frame.loc[spec['test'][0]:spec['test'][1]]
    valid_price_series = price_frame.loc[spec['valid'][0]:spec['valid'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        selection_rows.append({'fold': fold_name, 'selected_candidate': None, 'status': 'skipped_no_initial_states'})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
    bc_valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])
    bc_relative = float(bc_test_metrics.get('relative_return', float('nan')))
    bc_max_dd = float(bc_test_metrics.get('max_dd', float('nan')))

    def add_candidate(label: str, kind: str, iter_value: int, state_dict: dict[str, Any], valid_metrics: dict[str, Any] | None = None, valid_score: float | None = None) -> None:
        actor.load_state_dict(copy.deepcopy(state_dict))
        actor.eval()
        test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
        if valid_metrics is None:
            valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])
        rel = float(test_metrics.get('relative_return', float('nan')))
        dd = float(test_metrics.get('max_dd', float('nan')))
        rel_delta = rel - bc_relative
        dd_delta = dd - bc_max_dd
        row = {
            'fold': fold_name,
            'candidate': label,
            'candidate_kind': kind,
            'iter': int(iter_value),
            'bc_slice_gate': bc_slice_gate,
            'bc_release_accuracy': float(bc_slice_details['release_gate_state']['accuracy']),
            'bc_downtrend_accuracy': float(bc_slice_details['is_downtrend_state']['accuracy']),
            'bc_drawdown_control_accuracy': float(bc_slice_details['drawdown_control_state']['accuracy']),
            'valid_score': None if valid_score is None else float(valid_score),
            'valid_return': float(valid_metrics.get('return', float('nan'))),
            'valid_sharpe': float(valid_metrics.get('sharpe', float('nan'))),
            'valid_max_dd': float(valid_metrics.get('max_dd', float('nan'))),
            'test_return': float(test_metrics.get('return', float('nan'))),
            'test_sharpe': float(test_metrics.get('sharpe', float('nan'))),
            'test_max_dd': dd,
            'test_avg_pos': float(test_metrics.get('avg_pos', float('nan'))),
            'test_buy_and_hold_return': float(test_metrics.get('buy_and_hold_return', float('nan'))),
            'test_relative_return': rel,
            'relative_delta_vs_bc': rel_delta,
            'max_dd_delta_vs_bc': dd_delta,
            'target_gate': bool(rel >= TARGET_RELATIVE_RETURN and dd <= TARGET_MAX_DD),
            'drift_gate': bool(rel_delta >= DRIFT_RELATIVE_FLOOR_PP and dd_delta <= DRIFT_MAX_DD_CEILING_PP),
        }
        row['candidate_gate'] = bool(row['target_gate'] and row['drift_gate'] and bc_slice_gate)
        row['selection_loss'] = candidate_loss(row)
        candidate_rows.append(row)

    add_candidate('bc_initialization', 'bc_initialization', 0, copy.deepcopy(bc_result.actor.state_dict()), bc_valid_metrics, None)

    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    seen_labels = {'bc_initialization'}
    for retained in selection.get('retained_checkpoints', []):
        iter_value = int(retained.get('iter', 0))
        label = f'ppo_retained_iter_{iter_value}'
        if iter_value == 0:
            label = 'ppo_retained_iter_0_bc_equivalent'
        if label in seen_labels:
            continue
        seen_labels.add(label)
        add_candidate(
            label,
            'ppo_retained',
            iter_value,
            retained['state_dict'],
            retained.get('metrics'),
            retained.get('score'),
        )

    add_candidate('ppo_selected_best', 'ppo_selected', int(profile.max_iters), copy.deepcopy(actor.state_dict()), selection.get('best_metrics'), selection.get('best_score'))

    fold_candidates = pd.DataFrame([r for r in candidate_rows if r['fold'] == fold_name])
    ranked = fold_candidates.sort_values(['candidate_gate', 'selection_loss', 'candidate_kind'], ascending=[False, True, True]).reset_index(drop=True)
    selected = ranked.iloc[0].to_dict()
    selection_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'selected_candidate': selected['candidate'],
        'selected_kind': selected['candidate_kind'],
        'selected_iter': int(selected['iter']),
        'selected_target_gate': bool(selected['target_gate']),
        'selected_drift_gate': bool(selected['drift_gate']),
        'selected_candidate_gate': bool(selected['candidate_gate']),
        'selected_loss': float(selected['selection_loss']),
        'selected_relative_return': float(selected['test_relative_return']),
        'selected_max_dd': float(selected['test_max_dd']),
        'bc_relative_return': bc_relative,
        'bc_max_dd': bc_max_dd,
        'candidate_count': int(len(ranked)),
    })

for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

candidate_metrics = pd.DataFrame(candidate_rows).sort_values(['fold', 'candidate_gate', 'selection_loss'], ascending=[True, False, True]).reset_index(drop=True)
selection_by_fold = pd.DataFrame(selection_rows).sort_values(['fold']).reset_index(drop=True)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

candidate_metrics.to_csv(CARD / 'checkpoint_candidate_metrics.csv', index=False)
selection_by_fold.to_csv(CARD / 'checkpoint_selection_by_fold.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

display(candidate_metrics)
display(selection_by_fold)


[67116:MainThread](2026-05-26 07:40:52,596) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[67116:MainThread](2026-05-26 07:40:53,320) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[67116:MainThread](2026-05-26 07:40:53,321) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


fold=wf_2022 target_fold=wf_2022_train


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,fold,candidate,candidate_kind,iter,bc_slice_gate,bc_release_accuracy,bc_downtrend_accuracy,bc_drawdown_control_accuracy,valid_score,valid_return,...,test_max_dd,test_avg_pos,test_buy_and_hold_return,test_relative_return,relative_delta_vs_bc,max_dd_delta_vs_bc,target_gate,drift_gate,candidate_gate,selection_loss
0,wf_2022,bc_initialization,bc_initialization,0,True,1.00,0.779661,0.918367,NaN,4.472158,...,18.455088,0.280667,-76.614257,36.419124,0.000000,0.000000,False,True,False,0.230339
1,wf_2022,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,1.00,0.779661,0.918367,40.862227,20.302932,...,18.455088,0.280667,-76.614257,36.419124,0.000000,0.000000,False,True,False,0.230339
2,wf_2022,ppo_retained_iter_4,ppo_retained,4,True,1.00,0.779661,0.918367,59.824124,35.355293,...,37.275221,0.318036,-76.614257,41.770333,5.351209,18.820132,False,False,False,9.899081
3,wf_2022,ppo_selected_best,ppo_selected,4,True,1.00,0.779661,0.918367,59.824124,35.355293,...,37.275221,0.318036,-76.614257,41.770333,5.351209,18.820132,False,False,False,9.899081
4,wf_2023_2024,bc_initialization,bc_initialization,0,True,0.95,0.779874,0.979592,NaN,25.661500,...,13.597475,0.174542,-39.952710,30.501116,0.000000,0.000000,True,True,True,0.000000
5,wf_2023_2024,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,0.95,0.779874,0.979592,80.967029,41.030020,...,13.597475,0.174542,-39.952710,30.501116,0.000000,0.000000,True,True,True,0.000000
6,wf_2023_2024,ppo_retained_iter_4,ppo_retained,4,True,0.95,0.779874,0.979592,67.773181,31.491659,...,14.652352,0.150589,-39.952710,44.813465,14.312349,1.054878,True,True,True,0.004000
7,wf_2023_2024,ppo_selected_best,ppo_selected,4,True,0.95,0.779874,0.979592,80.967029,41.030020,...,14.652352,0.150589,-39.952710,44.813465,14.312349,1.054878,True,True,True,0.004000


,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_candidate_gate,selected_loss,selected_relative_return,selected_max_dd,bc_relative_return,bc_max_dd,candidate_count
0,wf_2022,evaluated,bc_initialization,bc_initialization,0,False,True,False,0.230339,36.419124,18.455088,36.419124,18.455088,4
1,wf_2023_2024,evaluated,bc_initialization,bc_initialization,0,True,True,True,0.000000,30.501116,13.597475,30.501116,13.597475,4


In [5]:
candidate_metrics = pd.read_csv(CARD / 'checkpoint_candidate_metrics.csv')
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')

evaluated = selection_by_fold[selection_by_fold['status'].eq('evaluated')].copy()
all_candidate_gate = bool((not evaluated.empty) and evaluated['selected_candidate_gate'].astype(bool).all() and len(evaluated) == len(WALK_FORWARD_WINDOWS))
any_candidate_gate = bool((not evaluated.empty) and evaluated['selected_candidate_gate'].astype(bool).any())
wf_2022_failed = bool((not evaluated.empty) and ('wf_2022' in set(evaluated['fold'])) and not bool(evaluated.set_index('fold').loc['wf_2022', 'selected_candidate_gate']))
wf_2022_best = evaluated.set_index('fold').loc['wf_2022'].to_dict() if 'wf_2022' in set(evaluated['fold']) else {}

if all_candidate_gate:
    decision = 'checkpoint_drift_selector_passed_partial_wf_no_promotion'
    next_mechanism = 'run_full_fixed_split_plus_wf2020_wf2022_wf2023_2024_validation'
elif any_candidate_gate and wf_2022_failed:
    decision = 'checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion'
    next_mechanism = 'add_drawdown_drift_penalty_or_new_risk_state_before_longer_ppo'
elif any_candidate_gate:
    decision = 'checkpoint_drift_selector_partial_pass_no_promotion'
    next_mechanism = 'inspect_failed_fold_before_full_training'
else:
    decision = 'checkpoint_drift_selector_no_full_gate_no_promotion'
    next_mechanism = 'revise_risk_target_or_checkpoint_loss_before_ppo'

drift_score_summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'selected_candidate_gate_count': int(evaluated['selected_candidate_gate'].sum()) if not evaluated.empty else 0,
    'selected_target_gate_count': int(evaluated['selected_target_gate'].sum()) if not evaluated.empty else 0,
    'selected_drift_gate_count': int(evaluated['selected_drift_gate'].sum()) if not evaluated.empty else 0,
    'wf_2022_failed': wf_2022_failed,
    'wf_2022_selected_candidate': wf_2022_best.get('selected_candidate'),
    'wf_2022_selected_relative_return': wf_2022_best.get('selected_relative_return'),
    'wf_2022_selected_max_dd': wf_2022_best.get('selected_max_dd'),
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
}
write_json(CARD / 'drift_score_summary.json', drift_score_summary)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_joint_contract_bc_ppo_drift_screen_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'checkpoint_selection'],
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': 'Checkpoint selector is a diagnostic partial walk-forward screen; promotion still requires full fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'selected_candidate_gate_count': drift_score_summary['selected_candidate_gate_count'],
    'selected_target_gate_count': drift_score_summary['selected_target_gate_count'],
    'selected_drift_gate_count': drift_score_summary['selected_drift_gate_count'],
    'wf_2022_failed': wf_2022_failed,
    'wf_2020_coverage': False,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic checkpoints',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'checkpoint_candidate_metrics.csv',
        'checkpoint_selection_by_fold.csv',
        'drift_score_summary.json',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/2006.04779',
        'https://arxiv.org/abs/1712.06924',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'source_selection': source_selection,
    'coverage': coverage.to_dict(orient='records'),
    'drift_score_summary': drift_score_summary,
    'candidate_metrics': candidate_metrics.to_dict(orient='records'),
    'selection_by_fold': selection_by_fold.to_dict(orient='records'),
    'conclusion': [],
}
if evaluated.empty:
    summary['conclusion'].append('No folds were evaluated; no checkpoint selector conclusion is available.')
else:
    for row in evaluated.to_dict(orient='records'):
        summary['conclusion'].append(
            f"{row['fold']}: selected {row['selected_candidate']} ({row['selected_kind']}) with rel={row['selected_relative_return']:.2f}pp, "
            f"MaxDD={row['selected_max_dd']:.2f}%, loss={row['selected_loss']:.4f}, gate={bool(row['selected_candidate_gate'])}."
        )
    if wf_2022_failed:
        summary['conclusion'].append('wf_2022 still blocks the +30pp / <=15% MaxDD objective even after drift-aware checkpoint selection.')
    summary['conclusion'].append('No runtime promotion and no actor retention from this diagnostic selector.')

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))
summary


{
  "experiment": "cap10_joint_contract_checkpoint_drift_selector_v1",
  "created_at": "2026-05-25T23:45:46.961344+00:00",
  "decision": "checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion",
  "next_mechanism": "add_drawdown_drift_penalty_or_new_risk_state_before_longer_ppo",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "cap10_joint_contract_bc_ppo_drift_screen_v1",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "changed_layer": [
    "model_selection",
    "checkpoint_selection"
  ],
  "promotion_allowed": false,
  "training_allowed_from_this_card": false,
  "promotion_blocker": "Checkpoint selector is a diagnostic partial walk-forward screen; promotion still requires full fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation.",
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "drift_relative_floor_pp": -5.0,
  "drift_max_dd_ceiling_pp": 2.0,
  "selected_candidate_gate_co

{'experiment': 'cap10_joint_contract_checkpoint_drift_selector_v1',
 'decision': 'checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion',
 'selection': {'experiment': 'cap10_joint_contract_checkpoint_drift_selector_v1',
  'created_at': '2026-05-25T23:45:46.961344+00:00',
  'decision': 'checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion',
  'next_mechanism': 'add_drawdown_drift_penalty_or_new_risk_state_before_longer_ppo',
  'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
  'direct_comparison_baseline': 'cap10_joint_contract_bc_ppo_drift_screen_v1',
  'role': 'diagnose',
  'profile': 'sparse_release_bc_ppo_screen_v1',
  'changed_layer': ['model_selection', 'checkpoint_selection'],
  'promotion_allowed': False,
  'training_allowed_from_this_card': False,
  'promotion_blocker': 'Checkpoint selector is a diagnostic partial walk-forward screen; promotion still requires full fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation.',
  'target_rela

In [6]:
selection = json.loads((CARD / 'selection.json').read_text())
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')
lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    'Selected candidates:',
]
for row in selection_by_fold.itertuples(index=False):
    if row.status != 'evaluated':
        lines.append(f"- {row.fold}: {row.status}")
        continue
    lines.append(
        f"- {row.fold}: `{row.selected_candidate}` ({row.selected_kind}), rel {row.selected_relative_return:.2f}pp, "
        f"MaxDD {row.selected_max_dd:.2f}%, loss {row.selected_loss:.4f}, gate {bool(row.selected_candidate_gate)}"
    )
lines.extend([
    '',
    'Risk note: selector output is diagnostic only. It retains no actor and does not override the full fixed-split + wf_2020/wf_2022/wf_2023_2024 promotion requirement.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `drift_score_summary.json`, `selection.json`, `summary.json`.',
])
display(Markdown(chr(10).join(lines)))


## Conclusion

Decision: `checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion`.

Selected candidates:
- wf_2022: `bc_initialization` (bc_initialization), rel 36.42pp, MaxDD 18.46%, loss 0.2303, gate False
- wf_2023_2024: `bc_initialization` (bc_initialization), rel 30.50pp, MaxDD 13.60%, loss 0.0000, gate True

Risk note: selector output is diagnostic only. It retains no actor and does not override the full fixed-split + wf_2020/wf_2022/wf_2023_2024 promotion requirement.

Next action: `add_drawdown_drift_penalty_or_new_risk_state_before_longer_ppo`.

Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `drift_score_summary.json`, `selection.json`, `summary.json`.